# Knowledge Graphs and Semantic Technologies -- RDF tutorial

In this tutorial we'll learn the basics of interacting with RDF graphs with Python. We'll be using rdflib for this, a widely used Ptyhon library for RDF (all documentation can be found [here](https://rdflib.readthedocs.io/en/stable/index.html))

## Imports
These are the main classes and types we'll be using from rdflib

In [89]:
import sys
from rdflib import Graph, ConjunctiveGraph, Literal, BNode, Namespace, RDF, URIRef, RDFS
from rdflib.namespace import DC, FOAF
import pprint


## Loading data remotely and from files

rdflib accepts importing RDF data from a variety of sources, either locally from a file (including an extensive support of serializations), or remotely via a URI (this is a great way of checking practically if URIs return RDF according to the 3rd Linked Data principle).

A Graph object is always required to load triples.
**Note**: to load quads, and hence supporting named graphs, you'll need to use an instance of ConjunctiveGraph instead

**Exercise 1** 

For each step, use a different cell: 
1. create two graphs using rdflib:
    - and load one with triples from the site https://csarven.ca/ and/or http://www.w3.org/People/Berners-Lee/card 
    - load one with triples from ./data/ingredients.rdf. 

In [90]:
#TIP: look at the documentation of the rdflib library for how to LOAD and PARSE a graph - https://rdflib.readthedocs.io/en/stable/gettingstarted.html

#remote graph
g_remote = Graph()
g_remote.parse("http://www.w3.org/People/Berners-Lee/card")

print("# of triples:", len(g_remote))
print(g_remote.serialize(format="turtle")[:1000])


# of triples: 86
@prefix : <http://xmlns.com/foaf/0.1/> .
@prefix Be: <https://www.w3.org/People/Berners-Lee/> .
@prefix Pub: <https://timbl.com/timbl/Public/> .
@prefix blog: <http://dig.csail.mit.edu/breadcrumbs/blog/> .
@prefix card: <https://www.w3.org/People/Berners-Lee/card#> .
@prefix cc: <http://creativecommons.org/ns#> .
@prefix cert: <http://www.w3.org/ns/auth/cert#> .
@prefix con: <http://www.w3.org/2000/10/swap/pim/contact#> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix doap: <http://usefulinc.com/ns/doap#> .
@prefix geo1: <http://www.w3.org/2003/01/geo/wgs84_pos#> .
@prefix ldp: <http://www.w3.org/ns/ldp#> .
@prefix s: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema1: <http://schema.org/> .
@prefix sioc: <http://rdfs.org/sioc/ns#> .
@prefix solid: <http://www.w3.org/ns/solid/terms#> .
@prefix space: <http://www.w3.org/ns/pim/space#> .
@prefix vcard: <http://www.w3.org/2006/vcard/ns#> .
@prefix w3c: <http://ww

In [91]:
#local graph 

g_local = Graph()
g_local.parse("./data/ingredients.rdf")
print("# of triples:", len(g_local))

# of triples: 837


In [92]:
#loeading one triple
for s, p, o in g_local:
    print("sub:", s)
    print("pred:", p)
    print("obj:", o)
    break  

sub: http://purl.org/heals/ingredient/BrownSugar
pred: http://www.w3.org/2000/01/rdf-schema#label
obj: brown sugar


## Serialising and saving RDF graphs

There are different formats for storing RDF triples. Semantically, these mean the same, they differ only in their syntax. 


Use the function Graph.serialize(format). 

**Exercise 2**

1. serialise one of the graphs to the .ttl, .xml and .nt format, and print the first n lines to compare the syntax
1. save your graph in the turtle format to the ./data/ folder

In [93]:
#serialize the chosen graph
n = 5  # nr lines for comparison

ttl_str = g_local.serialize(format="turtle")
xml_str = g_local.serialize(format="xml")
nt_str  = g_local.serialize(format="nt")

print("Turtle = .ttl")
print("\n".join(ttl_str.splitlines()[:n]))

print("\n RDF/XML = .xml")
print("\n".join(xml_str.splitlines()[:n]))

print("\n N-Trioples = .nt")
print("\n".join(nt_str.splitlines()[:n]))

#save the graph in ttl format
g_local.serialize(destination="./data/ingredients.ttl", format="turtle")

Turtle = .ttl
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ind: <http://purl.org/heals/ingredient/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

 RDF/XML = .xml
<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF
   xmlns:dcterms="http://purl.org/dc/terms/"
   xmlns:owl="http://www.w3.org/2002/07/owl#"
   xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"

 N-Trioples = .nt
<http://purl.org/heals/ingredient/BrownSugar> <http://www.w3.org/2000/01/rdf-schema#label> "brown sugar" .
<http://purl.org/heals/ingredient/Tomato> <http://www.w3.org/2004/02/skos/core#scopeNote> "used as an ingredient" .
<http://purl.org/heals/ingredient/Gooey> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#NamedIndividual> .
<http://purl.org/heals/ingredient/Beef> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://purl.org/heals/food/Ingredient> .
<http

<Graph identifier=N9ab265aba3d84e4091d2295db8be064f (<class 'rdflib.graph.Graph'>)>

##  Merging graphs

Merging graphs can be done via sequential parsings or by the overloaded operator +

**Note:** Set-theoretic graph semantics apply

The Food knowledge graph FoodKG contains a graph of statements about ingredients, as well as a graph with statements about recipes. 

**Exercise 3**: 

1. load ./data/ingredients.rdf and ./data/ghostbusters.ttl into a single graph, either by sequential parsing or using the operator +.

2. count the number of statements in each graph, and the intersection of the two graphs. 

3. check whether the combined graph is connected (using graph.connected()) 

4. load ./data/ingredients.rdf and ./data/recipes.rdf into a single graph, either by sequential parsing or using the operator +. 

5. count the number of statements in each graph, and the intersection of the two graphs. 

6. check whether the combined graph is connected (using graph.connected()). Explain the result with respect to point 3! 

In [94]:
#look at rdflib documentation - Navigating Graphs

#1
g_ing = Graph()
g_ing.parse("./data/ingredients.rdf")   # if errors: format="xml"
g_ghost = Graph()
g_ghost.parse("./data/ghostbusters.ttl", format="turtle")
g_combined = g_ing + g_ghost

print("ing triples:", len(g_ing))
print("ghost triples:", len(g_ghost))
print("combi triples:", len(g_combined))

ing triples: 837
ghost triples: 52337
combi triples: 53174


In [95]:
# 2
ing_set = set(g_ing)
ghost_set = set(g_ghost)
intersection_1 = ing_set.intersection(ghost_set)
print("size of intersection:", len(intersection_1))

size of intersection: 0


In [96]:
#3
print("connected graphs?", g_combined.connected())

connected graphs? False


In [97]:
#4 
g_ing2 = Graph()
g_ing2.parse("./data/ingredients.rdf")
g_reci = Graph()
g_reci.parse("./data/recipes.rdf") 
g_ing_reci = g_ing2 + g_reci

print("\nIng:", len(g_ing2))
print("reci:", len(g_reci))
print("combi:", len(g_ing_reci))


Ing: 837
reci: 480
combi: 1299


In [98]:
#5
ing2_set = set(g_ing2)
reci_set = set(g_reci)
intersection_2 = ing2_set.intersection(reci_set)

print("size of interaction:", len(intersection_2))

size of interaction: 18


In [99]:
# 6
print("connected ing + reci graphs?", g_ing_reci.connected())

connected ing + reci graphs? False


Both graphs are not connected.
Ingredients and ghostbusters is not conencted because they do not share any related entities or nodes or URIs. In the beginning I thought that ingredients and recipes will be connected because recipes could reference ingredients and this way creating the path between them. However, they are not. The reason for that could be that recipes use different identifiers than ingredients or even only literals instead of ingredient URIs, therefore there are no links bringing the data. 

## Namespaces 

Remind yourself what namespaces are. 

In RDFLib, the namespace module defines many common namespaces such as RDF, RDFS, OWL, FOAF, SKOS, etc., but you can also easily add URIs within a different namespace:


In [100]:
TEACH = Namespace("http://linkedscience.org/teach/ns#")
TEACH.Teacher

rdflib.term.URIRef('http://linkedscience.org/teach/ns#Teacher')

Check out the specification to see which other terms are used within the TEACH namespace. http://linkedscience.org/teach/ns/#sec-specification. 
You can use a NamespaceManager to bind a prefix to a namespace: 

In [101]:
g = Graph()
g.namespace_manager.bind('TEACH', URIRef('http://linkedscience.org/teach/ns#'))
TEACH.Teacher.n3(g.namespace_manager)

'TEACH:Teacher'

In [102]:
KRW = Namespace("http://krw.vu.nl/data#")

#creating individuals within your namespace
KRW.Teacher
KRW.Student

rdflib.term.URIRef('http://krw.vu.nl/data#Student')

**Exercise 4:**
1. create your own namespace (can be made up) 

In [103]:
# my own namespace 
kg_st = Namespace("https://studiegids.vu.nl/en/courses/2025-2026/XM_0147")



## Creating RDF triples

Triples are added to the graph with the function Graph.add()

The parameter is a triple given in a Python **tuple** (subject, predicate, object)

Notice the namespace convenience syntax!

**Exercise 5:** 

1. create a new graph and add triples (~10) within your made-up namespace using Graph.add(). These triples can be about anything, for instance ingredients or recipes. Make sure they include the predicates RDF.type, RDFS.label and RDFS.subClassOf

2. open yourRDF.ttl, and write your triples out by hand in a syntax of your choice (turtle is recommended, notice the file extension!). Load the triples here with rdflib. 

In [104]:
#create graph
my_graph = Graph()

#example namespace
kg_st = Namespace("https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#")
my_graph.bind("kg", kg_st)


# add triples using store's add method
#classes
my_graph.add((kg_st.Course, RDF.type, RDFS.Class))
my_graph.add((kg_st.MasterCourse, RDF.type, RDFS.Class))
my_graph.add((kg_st.MasterCourse, RDFS.subClassOf, kg_st.Course))
#course instance
my_graph.add((kg_st.XM_0147, RDF.type, kg_st.MasterCourse))
my_graph.add((kg_st.XM_0147, RDFS.label, Literal("Knowledge Graphs and Semantic Technologies")))
#simple properties
my_graph.add((kg_st.hasECTS, RDF.type, RDF.Property))
my_graph.add((kg_st.hasLanguage, RDF.type, RDF.Property))
# values from the website
my_graph.add((kg_st.XM_0147, kg_st.hasECTS, Literal("6 EC")))
my_graph.add((kg_st.XM_0147, kg_st.hasLanguage, Literal("English")))

#print("nr of triples:", len(my_graph))
#print(my_graph.serialize(format="turtle"))

# save the graph to destination in ttl format - myRDF.ttl (look at RDFLib documentation - Loading and saving RDF)
my_graph.serialize(destination="./data/myRDF.ttl", format="turtle")
print("Saved ./data/myRDF.ttl")

# load the saved graph and print it in ttl format
g_loaded = Graph()
g_loaded.parse("./data/myRDF.ttl", format="turtle")
print("Loaded triples:", len(g_loaded))
print(g_loaded.serialize(format="turtle"))

Saved ./data/myRDF.ttl
Loaded triples: 9
@prefix kg: <https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

kg:Course a rdfs:Class .

kg:MasterCourse a rdfs:Class ;
    rdfs:subClassOf kg:Course .

kg:XM_0147 a kg:MasterCourse ;
    rdfs:label "Knowledge Graphs and Semantic Technologies" ;
    kg:hasECTS "6 EC" ;
    kg:hasLanguage "English" .

kg:hasECTS a rdf:Property .

kg:hasLanguage a rdf:Property .




## Navigating graphs

rdflib uses iterators to navigate Graphs. The methods for navigating subjects, predicates and objects are Graph.subjects, Graph.predicates, Graph.objects

**Exercise 6:**

1. print all the triples in yourRDF.ttl
2. print all subjects in yourRDF.ttl
3. print all predicates in yourRDF.ttl
4. print all objects in yourRDF.ttl


In [105]:
#TIP you have to loop in the graph 

g = Graph()
g.parse("./data/myRDF.ttl", format="turtle")
print("triples loaded:", len(g))

triples loaded: 9


In [106]:
# 1 all triples
for s, p, o in g:
    print(s, p, o)

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasECTS 6 EC
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 http://www.w3.org/2000/01/rdf-schema#label Knowledge Graphs and Semantic Technologies
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#Course http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2000/01/rdf-schema#Class
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasECTS http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse http://www.w3.org/2000/01/rdf-schema#subClassOf https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#Course
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM

In [107]:
# 2 all subjects
for s in g.subjects():
    print(s) 

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#Course
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasECTS
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasLanguage


In [108]:
# 3 all predicates
for p in g.predicates():
    print(p)

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasECTS
http://www.w3.org/2000/01/rdf-schema#label
http://www.w3.org/1999/02/22-rdf-syntax-ns#type
http://www.w3.org/1999/02/22-rdf-syntax-ns#type
http://www.w3.org/2000/01/rdf-schema#subClassOf
http://www.w3.org/1999/02/22-rdf-syntax-ns#type
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasLanguage
http://www.w3.org/1999/02/22-rdf-syntax-ns#type
http://www.w3.org/1999/02/22-rdf-syntax-ns#type


In [109]:
# 4 all objects
for o in g.objects():
    print(o)

6 EC
Knowledge Graphs and Semantic Technologies
http://www.w3.org/2000/01/rdf-schema#Class
http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#Course
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse
English
http://www.w3.org/2000/01/rdf-schema#Class
http://www.w3.org/1999/02/22-rdf-syntax-ns#Property


We can also filter the subjects, predicates and objects we want to retrieve, and match their values like in a database "join" operation


**Exercise 7:**

1. print all subject types in yourRDF.ttl
2. print all subject labels yourRDF.ttl

In [110]:
#1
for subj, typ in g.subject_objects(RDF.type):
    print(subj, "rdf:type", typ)

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#Course rdf:type http://www.w3.org/2000/01/rdf-schema#Class
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse rdf:type http://www.w3.org/2000/01/rdf-schema#Class
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 rdf:type https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasECTS rdf:type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasLanguage rdf:type http://www.w3.org/1999/02/22-rdf-syntax-ns#Property


In [111]:
#2
for subj, label in g.subject_objects(RDFS.label):
    print(subj, "rdfs:label", label)

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 rdfs:label Knowledge Graphs and Semantic Technologies


### Basic triple matching (almost querying!)

We use method Graph.triples and a Python tuple that acts as a mask for specifying our criteria

**Exercise 8:**

1. check whether a triple is in your graph -> print true or false
2. print all triples related to a certain subject in your graph
3. print all triples related to a certain object in your graph

In [112]:
#1 
triple = (
    kg_st.XM_0147,
    RDFS.label,
    Literal("Knowledge Graphs and Semantic Technologies")
)

print(triple in g)

True


In [113]:
#2
for s, p, o in g.triples((kg_st.XM_0147, None, None)):
    print(s, p, o)

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 http://www.w3.org/2000/01/rdf-schema#label Knowledge Graphs and Semantic Technologies
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasECTS 6 EC
https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#hasLanguage English


In [114]:
#3 triples with object MasterCourse
for s, p, o in g.triples((None, None, kg_st.MasterCourse)):
    print(s, p, o)

https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#XM_0147 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://studiegids.vu.nl/en/courses/2025-2026/XM_0147#MasterCourse


## Restaurant Exercise - Part 1

You are a chef in a restaurant, and you need to serve someone that is gluten intolerant. 

1. load the ./data/recipes.rdf and ./data/ingredients.rdf datasets in one graph
2. query your graph (as we did in previous exercises) to retrieve all recipes without gluten
3. query your graph for all recipes that you can make for your gluten intolerant guest. 
4. the guest asks you whether there are more options. Can you find the recipes for which an ingredient with gluten can be replaced, solely using pattern matching? (Hint: you need to write multiple of these pattern matching queries, and check the predicate __substitutesFor__) 
5. another guest is allergic to pecan nuts, which recipes could you serve them (including those for which pecan nuts can be replaced) 

**Note that this is a bit tedious: later on, we will be querying more complicated patterns with SPARQL!**

In [115]:
#1
g_food = Graph()
g_food.parse("./data/ingredients.rdf", format="xml")
g_food.parse("./data/recipes.rdf", format="xml")

print("total triples", len(g_food))

total triples 1299


In [116]:
#2
WTM = Namespace("http://purl.org/heals/food/")
IND = Namespace("http://purl.org/heals/ingredient/")

HAS_GLUTEN = WTM.hasGluten
SUBSTITUTES_FOR = WTM.substitutesFor

def is_ind_ingredient(node):
    return isinstance(node, URIRef) and str(node).startswith(str(IND))

pred_counts = {}
for s, p, o in g_food:
    if is_ind_ingredient(o):
        pred_counts[p] = pred_counts.get(p, 0) + 1

top = sorted(pred_counts.items(), key=lambda x: x[1], reverse=True)[:15]
print("top predicates that point to ind:")
for p, c in top:
    print(c, p)

HAS_INGREDIENT = top[0][0] if top else None
print("\nHAS_INGREDIENT predicate:", HAS_INGREDIENT)

top predicates that point to ind:
186 http://purl.org/heals/food/hasIngredient
12 http://purl.org/heals/food/substitutesFor
3 http://purl.org/heals/food/hasTexture
2 http://purl.org/heals/food/dislikes
1 http://purl.org/heals/food/forbids
1 http://www.w3.org/2002/07/owl#versionIRI
1 http://purl.org/heals/food/isAllergicTo

HAS_INGREDIENT predicate: http://purl.org/heals/food/hasIngredient


In [117]:
#3
# ing -> hasGluten 
ingredient_gluten = {}
for ing, gluten_val in g_food.subject_objects(HAS_GLUTEN):
    ingredient_gluten[ing] = str(gluten_val).lower() == "true"

print("ing with hasGluten info:", len(ingredient_gluten))

# recipe -> set(ing)
recipe_to_ings = {}
if HAS_INGREDIENT is None:
    raise ValueError("could not find a predicate that links recipes to ing.")

for recipe, ing in g_food.subject_objects(HAS_INGREDIENT):
    if is_ind_ingredient(ing):
        recipe_to_ings.setdefault(recipe, set()).add(ing)

print("rcipes with ing links:", len(recipe_to_ings))

ing with hasGluten info: 74
rcipes with ing links: 22


In [118]:
#4
gluten_free_recipes = set()

for r, ings in recipe_to_ings.items():
    if all(ingredient_gluten.get(i) is False for i in ings if i in ingredient_gluten):
        if not any(ingredient_gluten.get(i) is True for i in ings):
            gluten_free_recipes.add(r)

print("gluten-free recipes:", len(gluten_free_recipes))
for r in list(gluten_free_recipes)[:30]:
    print(r)

gluten-free recipes: 12
http://purl.org/heals/ingredient/PotRoastWithVegetables
http://purl.org/heals/ingredient/BananaBlueberryAlmondFlourMuffin
http://purl.org/heals/ingredient/BeefStew
http://purl.org/heals/ingredient/BeefNilaga
http://purl.org/heals/ingredient/SaucyShepherdPie
http://purl.org/heals/ingredient/SmotheredChickenBreast
http://purl.org/heals/ingredient/GrilledChickenKabob
http://purl.org/heals/ingredient/CornedBeefHash
http://purl.org/heals/ingredient/FlourlessCoconutAndAlmondCake
http://purl.org/heals/ingredient/BakedChickenTender
http://purl.org/heals/ingredient/GlutenFreeCoconutCake
http://purl.org/heals/ingredient/BraisedBalsamicChicken


In [119]:
#5
print("serveable recipes for gluten intolerant guest:", len(gluten_free_recipes))

serveable recipes for gluten intolerant guest: 12


In [120]:
#6
subs_for = {}
for sub, target in g_food.subject_objects(SUBSTITUTES_FOR):
    subs_for.setdefault(target, set()).add(sub)

def has_gluten_free_substitute(ing):
    for sub in subs_for.get(ing, set()):
        if ingredient_gluten.get(sub) is False:
            return True
    return False

replaceable_gluten_recipes = set()

for r, ings in recipe_to_ings.items():
    gluten_ings = [i for i in ings if ingredient_gluten.get(i) is True]
    if not gluten_ings:
        continue  

    if all(has_gluten_free_substitute(i) for i in gluten_ings):
        replaceable_gluten_recipes.add(r)

print("recipes with gluten but fully replaceable:", len(replaceable_gluten_recipes))
for r in list(replaceable_gluten_recipes)[:30]:
    print(r)

all_options_for_gluten_guest = gluten_free_recipes | replaceable_gluten_recipes
print("\ntotal options for gluten intolerant guest (incl. replaceable):", len(all_options_for_gluten_guest))

recipes with gluten but fully replaceable: 4
http://purl.org/heals/ingredient/AlmondBiscotti
http://purl.org/heals/ingredient/Brownies
http://purl.org/heals/ingredient/BananaBread
http://purl.org/heals/ingredient/WhiteBread

total options for gluten intolerant guest (incl. replaceable): 16


In [121]:
#7
pecans = set()
for ing in set(g_food.subjects()):
    if isinstance(ing, URIRef) and str(ing).endswith("/Pecan"):
        pecans.add(ing)

for ing, lab in g_food.subject_objects(RDFS.label):
    if "pecan" in str(lab).lower():
        pecans.add(ing)

print("pecan ingredient nodes:", pecans)

pecan ingredient nodes: {rdflib.term.URIRef('http://purl.org/heals/ingredient/Pecan')}


## HI ontology exploration

In your project, you will be working with a Hybrid Intelligence (HI) ontology. This is an opportunity for you to get acquainted with its structure. Applying the skills from the exercises above perform the following actions:

1. Load the HI ontology from the data folder (hi_ontology.ttl) with RDFlib.
2. Create an "HI" Namespace.
3. Count the number of triples.
4. List all subjects.
5. List all predicates.
6. List all pairs of subjects and their corresponding objects linked by a rdf:type predicate.